In [1]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#pytorch dataset
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [3]:
def load_and_tokenisation(dataset_path):
  # ============================================================================
  # 2. Load and Prepare Data
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATA LOADING")
  print("=" * 60)

  df = pd.read_csv(dataset_path).dropna()
  print(f"✓ Dataset loaded: {len(df)} rows")
  print(f"✓ Columns: {df.columns.tolist()}")
  print(f"\nLabel distribution:")
  print(df['label'].value_counts())

  # ============================================================================
  # 3. Train/Validation/Test Split
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATA SPLITTING")
  print("=" * 60)

  # First split: 70% train, 30% temp (for val + test)
  train_texts, temp_texts, train_labels, temp_labels = train_test_split(
      df['combined_text'].tolist(),
      df['label'].tolist(),
      test_size=0.3, # Changed to 0.3 to get 70% train
      random_state=42,
      stratify=df['label']
  )

  # Second split: 50% validation, 50% test from the 30% temp data (which means 15% val, 15% test from total)
  val_texts, test_texts, val_labels, test_labels = train_test_split(
      temp_texts,
      temp_labels,
      test_size=0.5,
      random_state=42,
      stratify=temp_labels
  )

  print(f"✓ Training samples: {len(train_texts)}")
  print(f"✓ Validation samples: {len(val_texts)}")
  print(f"✓ Test samples: {len(test_texts)}")

  # ============================================================================
  # 4. Tokenization
  # ============================================================================
  print("\n" + "=" * 60)
  print("TOKENIZATION")
  print("=" * 60)

  tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  print("✓ Tokenizer loaded")

  print("Tokenizing training data...")
  train_encodings = tokenizer(
      train_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Training data tokenized")

  print("Tokenizing validation data...")
  val_encodings = tokenizer(
      val_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Validation data tokenized")

  print("Tokenizing test data...")
  test_encodings = tokenizer(
      test_texts,
      truncation=True,
      padding=True,
      max_length=128
  )
  print("✓ Test data tokenized")

  # ============================================================================
  # 5. Create PyTorch Dataset
  # ============================================================================
  print("\n" + "=" * 60)
  print("DATASET CREATION")
  print("=" * 60)

  train_dataset = FakeNewsDataset(train_encodings, train_labels)
  val_dataset = FakeNewsDataset(val_encodings, val_labels)
  test_dataset = FakeNewsDataset(test_encodings, test_labels)

  print(f"✓ Train dataset: {len(train_dataset)} samples")
  print(f"✓ Validation dataset: {len(val_dataset)} samples")
  print(f"✓ Test dataset: {len(test_dataset)} samples")

  return train_dataset, val_dataset, test_dataset, tokenizer

In [4]:
def load_model():
  # ============================================================================
  # 1. Setup Device (TPU/CUDA/CPU)
  # ============================================================================
  print("=" * 60)
  print("DEVICE SETUP")
  print("=" * 60)

  # Check for TPU first
  try:
      import torch_xla.core.xla_model as xm
      device = xm.xla_device()
      print(f"✓ Using TPU: {device}")
      use_tpu = True
  except ImportError:
      use_tpu = False
      if torch.cuda.is_available():
          device = torch.device("cuda")
          print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
          print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
      else:
          device = torch.device("cpu")
          print("⚠ Using CPU (Training will be slow!)")

  # ============================================================================
  # 2. Load Pre-trained BERT Model
  # ============================================================================
  print("\n" + "=" * 60)
  print("MODEL LOADING")
  print("=" * 60)

  model = BertForSequenceClassification.from_pretrained(
      'bert-base-uncased',
      num_labels=2
  )

  if not use_tpu:
      model.to(device)
      print(f"✓ BERT model loaded and moved to {device}")
  else:
      print("✓ BERT model loaded (TPU will handle device placement)")

  return model, device, use_tpu

In [5]:
# ============================================================================
# 7. Define Metrics
# ============================================================================
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [6]:
def finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics_fn, use_tpu, device):
  print("\n" + "=" * 60)
  print("TRAINING CONFIGURATION")
  print("=" * 60)

  # 8. Define Training Arguments
  training_args = TrainingArguments(
      output_dir='./results',
      num_train_epochs=3,
      per_device_train_batch_size=32,
      per_device_eval_batch_size=64,
      warmup_steps=500,
      weight_decay=0.01,
      logging_dir='./logs',
      logging_steps=50,
      eval_strategy="steps",
      eval_steps=1000,
      save_strategy="steps",
      save_steps=1000,
      save_total_limit=2,
      load_best_model_at_end=True,
      metric_for_best_model="f1",
      greater_is_better=True,
      optim="adamw_torch",
      fp16=False,
      dataloader_num_workers=0,
      report_to="none",
  )

  print("Training configuration:")
  print(f"  - Epochs: {training_args.num_train_epochs}")
  print(f"  - Train batch size: {training_args.per_device_train_batch_size}")
  print(f"  - Eval batch size: {training_args.per_device_eval_batch_size}")
  print(f"  - FP16: {training_args.fp16}")
  print(f"  - Device: {'TPU' if use_tpu else device}")


  print("\n" + "=" * 60)
  print("TRAINER INITIALIZATION")
  print("=" * 60)

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset, # The eval_dataset here is for validation during training
      compute_metrics=compute_metrics_fn
  )
  print("✓ Trainer initialized")

  print("\n" + "=" * 60)
  print("TRAINING START")
  print("=" * 60)

  trainer.train()
  print("\n✓ Training completed!")

  print("\n" + "=" * 60)
  print("VALIDATION SET EVALUATION")
  print("=" * 60)

  val_results = trainer.evaluate()
  print("Validation Results:")
  for key, value in val_results.items():
      print(f"  {key}: {value:.4f}")

  print("\n" + "=" * 60)
  print("TEST SET EVALUATION")
  print("=" * 60)

  test_results = trainer.evaluate(eval_dataset=test_dataset)
  print("Test Results:")
  for key, value in test_results.items():
      print(f"  {key}: {value:.4f}")

In [7]:
def save_model(model, tokenizer, output_path, use_tpu, device):
  print("\n" + "=" * 60)
  print("MODEL SAVING")
  print("=" * 60)

  output_dir = output_path

  # Move model to CPU before saving if it's on TPU
  if use_tpu:
      model.to("cpu")
      print("Moved model to CPU for saving.")

  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)
  print(f"✓ Model saved to {output_dir}")

  # Move model back to original device after saving
  if use_tpu:
      model.to(device)
      print("Moved model back to TPU.")

  print("\n" + "=" * 60)
  print("BERT FINE-TUNING COMPLETE! 🎉")
  print("=" * 60)

In [8]:
def train_loop(dataset_path, output_path):
  train_dataset, val_dataset, test_dataset, tokenizer = load_and_tokenisation(dataset_path)
  model, device, use_tpu = load_model()
  finetuning(model, train_dataset, val_dataset, test_dataset, compute_metrics, use_tpu, device)
  save_model(model, tokenizer, output_path, use_tpu, device)

# WELFake Dataset

In [ ]:
train_loop("/content/drive/MyDrive/datasets/WELFake_Dataset_processed.csv", "/content/drive/MyDrive/bert_models/WELFake")


DATA LOADING
✓ Dataset loaded: 72124 rows
✓ Columns: ['Unnamed: 0', 'combined_text', 'label']

Label distribution:
label
1    37096
0    35028
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 50486
✓ Validation samples: 10819
✓ Test samples: 10819

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 50486 samples
✓ Validation dataset: 10819 samples
✓ Test dataset: 10819 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.034600,0.043306,0.986320,0.986779,0.981169,0.992453
2000,0.023300,0.032718,0.990849,0.991067,0.995288,0.986882
3000,0.013100,0.035653,0.990942,0.991245,0.985610,0.996945
4000,0.009100,0.030615,0.994362,0.994511,0.996035,0.992992


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0306
  eval_accuracy: 0.9944
  eval_f1: 0.9945
  eval_precision: 0.9960
  eval_recall: 0.9930
  eval_runtime: 6.4205
  eval_samples_per_second: 847.2910
  eval_steps_per_second: 13.2390
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0354
  eval_accuracy: 0.9930
  eval_f1: 0.9932
  eval_precision: 0.9939
  eval_recall: 0.9925
  eval_runtime: 6.5312
  eval_samples_per_second: 832.9260
  eval_steps_per_second: 13.0140
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/WELFake
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# FakeNewsNet

In [ ]:
train_loop("/content/drive/MyDrive/datasets/FakeNewsNet_Dataset_processed.csv", "/content/drive/MyDrive/bert_models/FakeNewsNet")


DATA LOADING
✓ Dataset loaded: 23193 rows
✓ Columns: ['Unnamed: 0', 'combined_text', 'label']

Label distribution:
label
1    17439
0     5754
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 16235
✓ Validation samples: 3479
✓ Test samples: 3479

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 16235 samples
✓ Validation dataset: 3479 samples
✓ Test dataset: 3479 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.297800,0.332265,0.863754,0.912125,0.885529,0.940367


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.3323
  eval_accuracy: 0.8638
  eval_f1: 0.9121
  eval_precision: 0.8855
  eval_recall: 0.9404
  eval_runtime: 1.6116
  eval_samples_per_second: 1092.1140
  eval_steps_per_second: 17.3750
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.3311
  eval_accuracy: 0.8661
  eval_f1: 0.9137
  eval_precision: 0.8861
  eval_recall: 0.9430
  eval_runtime: 18.9543
  eval_samples_per_second: 92.8550
  eval_steps_per_second: 1.4770
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/FakeNewsNet
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# Fake News Detection Dataset

In [ ]:
train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_Dataset_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection_Dataset")


DATA LOADING
✓ Dataset loaded: 44258 rows
✓ Columns: ['Unnamed: 0', 'combined_text', 'label']

Label distribution:
label
0    22842
1    21416
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 30980
✓ Validation samples: 6639
✓ Test samples: 6639

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 30980 samples
✓ Validation dataset: 6639 samples
✓ Test dataset: 6639 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.010900,0.027321,0.994577,0.994370,0.999371,0.989418
2000,0.000500,0.011137,0.997741,0.997664,0.998441,0.996888


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0111
  eval_accuracy: 0.9977
  eval_f1: 0.9977
  eval_precision: 0.9984
  eval_recall: 0.9969
  eval_runtime: 3.8216
  eval_samples_per_second: 870.8460
  eval_steps_per_second: 13.6070
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0078
  eval_accuracy: 0.9982
  eval_f1: 0.9981
  eval_precision: 0.9981
  eval_recall: 0.9981
  eval_runtime: 3.8848
  eval_samples_per_second: 856.6690
  eval_steps_per_second: 13.3850
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/Fake_News_Detection_Dataset
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


# News Dataset

In [ ]:
train_loop("/content/drive/MyDrive/datasets/News_Dataset_processed.csv", "/content/drive/MyDrive/bert_models/News_Dataset_processed")


DATA LOADING
✓ Dataset loaded: 44889 rows
✓ Columns: ['Unnamed: 0', 'combined_text', 'label']

Label distribution:
label
1    23472
0    21417
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 31422
✓ Validation samples: 6733
✓ Test samples: 6734

TOKENIZATION
✓ Tokenizer loaded
Tokenizing training data...
✓ Training data tokenized
Tokenizing validation data...
✓ Validation data tokenized
Tokenizing test data...
✓ Test data tokenized

DATASET CREATION
✓ Train dataset: 31422 samples
✓ Validation dataset: 6733 samples
✓ Test dataset: 6734 samples
DEVICE SETUP
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipython-input-1204052393.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded (TPU will handle device placement)

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: TPU

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.000100,0.004210,0.999406,0.999432,0.999148,0.999716
2000,0.000400,0.002987,0.999554,0.999574,0.999149,1.000000


...


✓ Training completed!

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Results:
  eval_loss: 0.0030
  eval_accuracy: 0.9996
  eval_f1: 0.9996
  eval_precision: 0.9991
  eval_recall: 1.0000
  eval_runtime: 4.0207
  eval_samples_per_second: 843.6340
  eval_steps_per_second: 13.1820
  epoch: 3.0000

TEST SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Results:
  eval_loss: 0.0014
  eval_accuracy: 0.9999
  eval_f1: 0.9999
  eval_precision: 0.9997
  eval_recall: 1.0000
  eval_runtime: 10.7484
  eval_samples_per_second: 315.5830
  eval_steps_per_second: 4.9310
  epoch: 3.0000

MODEL SAVING
Moved model to CPU for saving.
✓ Model saved to /content/drive/MyDrive/bert_models/News_Dataset_processed
Moved model back to TPU.

BERT FINE-TUNING COMPLETE! 🎉


#aadyasingh55 Dataset

In [9]:
train_loop("/content/drive/MyDrive/datasets/aadyasingh55_Dataset_processed.csv", "/content/drive/MyDrive/bert_models/aadyasingh55_Dataset_processed")


DATA LOADING
✓ Dataset loaded: 40582 rows
✓ Columns: ['Unnamed: 0', 'combined_text', 'label']

Label distribution:
label
1    21924
0    18658
Name: count, dtype: int64

DATA SPLITTING
✓ Training samples: 28407
✓ Validation samples: 6087
✓ Test samples: 6088

TOKENIZATION


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ BERT model loaded and moved to cuda

TRAINING CONFIGURATION
Training configuration:
  - Epochs: 3
  - Train batch size: 32
  - Eval batch size: 64
  - FP16: False
  - Device: cuda

TRAINER INITIALIZATION
✓ Trainer initialized

TRAINING START


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1000,0.029100,0.028052,0.987186,0.988200,0.983143,0.993309
2000,0.012400,0.029965,0.989814,0.990563,0.991469,0.989659



✓ Training completed!

VALIDATION SET EVALUATION


Validation Results:
  eval_loss: 0.0300
  eval_accuracy: 0.9898
  eval_f1: 0.9906
  eval_precision: 0.9915
  eval_recall: 0.9897
  eval_runtime: 44.8448
  eval_samples_per_second: 135.7350
  eval_steps_per_second: 2.1410
  epoch: 3.0000

TEST SET EVALUATION
Test Results:
  eval_loss: 0.0257
  eval_accuracy: 0.9920
  eval_f1: 0.9925
  eval_precision: 0.9936
  eval_recall: 0.9915
  eval_runtime: 43.0362
  eval_samples_per_second: 141.4620
  eval_steps_per_second: 2.2310
  epoch: 3.0000

MODEL SAVING
✓ Model saved to /content/drive/MyDrive/bert_models/aadyasingh55_Dataset_processed

BERT FINE-TUNING COMPLETE! 🎉
